# 用于 MNIST 聚类与生成的 VAE

这个 notebook 的目标是探索一些关于变分自编码器（VAE）的近期工作。

我们将使用 MNIST 数据集和一个基础的 VAE 架构。


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.utils import save_image

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics.cluster import normalized_mutual_info_score

def show(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,0)), interpolation='nearest')
    
def plot_reconstruction(model, n=24):
    x,_ = next(iter(data_loader))
    x = x[:n,:,:,:].to(device)
    try:
        out, _, _, log_p = model(x.view(-1, image_size)) 
    except:
        out, _, _ = model(x.view(-1, image_size)) 
    x_concat = torch.cat([x.view(-1, 1, 28, 28), out.view(-1, 1, 28, 28)], dim=3)
    out_grid = torchvision.utils.make_grid(x_concat).cpu().data
    show(out_grid)

def plot_generation(model, n=24):
    with torch.no_grad():
        z = torch.randn(n, z_dim).to(device)
        out = model.decode(z).view(-1, 1, 28, 28)

    out_grid = torchvision.utils.make_grid(out).cpu()
    show(out_grid)

def plot_conditional_generation(model, n=8, fix_number=None):
    with torch.no_grad():
        matrix = np.zeros((n,n_classes))
        matrix[:,0] = 1

        if fix_number is None:
            final = matrix[:]
            for i in range(1,n_classes):
                final = np.vstack((final,np.roll(matrix,i)))
            #z = torch.randn(8*n_classes, z_dim).to(device)
            z = torch.randn(8, z_dim)
            z = z.repeat(n_classes,1).to(device)
            y_onehot = torch.tensor(final).type(torch.FloatTensor).to(device)
            out = model.decode(z,y_onehot).view(-1, 1, 28, 28)
        else:
            z = torch.randn(n, z_dim).to(device)
            y_onehot = torch.tensor(np.roll(matrix, fix_number)).type(torch.FloatTensor).to(device)
            out = model.decode(z,y_onehot).view(-1, 1, 28, 28)

    out_grid = torchvision.utils.make_grid(out).cpu()
    show(out_grid)

In [ ]:
# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 如果目录不存在则创建
sample_dir = 'samples'
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)

In [ ]:
data_dir = 'data'
# MNIST 数据集
dataset = torchvision.datasets.MNIST(root=data_dir,
                                     train=True,
                                     transform=transforms.ToTensor(),
                                     download=True)

# 数据加载器
data_loader = torch.utils.data.DataLoader(dataset=dataset,
                                          batch_size=128, 
                                          shuffle=True)

test_loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(data_dir, train=False, download=True, transform=transforms.ToTensor()),
    batch_size=10, shuffle=False)

# 变分自编码器

考虑一个潜变量模型：数据变量 $x\in \mathcal{X}$ 和潜变量 $z\in \mathcal{Z}$，$p(z,x) = p(z)p_\theta(x|z)$。给定数据 $x_1,\dots, x_n$，我们希望通过最大化边际对数似然来训练模型：
\begin{eqnarray*}
\mathcal{L} = \mathbf{E}_{p_d(x)}\left[\log p_\theta(x)\right]=\mathbf{E}_{p_d(x)}\left[\log \int_{\mathcal{Z}}p_{\theta}(x|z)p(z)dz\right],
  \end{eqnarray*}
  其中 $p_d$ 表示 $X$ 的经验分布：$p_d(x) =\frac{1}{n}\sum_{i=1}^n \delta_{x_i}(x)$。

 为了避免上面这个（通常很难算的）积分，变分方法的思想是改为最大化对数似然的一个下界：
  \begin{eqnarray*}
\mathcal{L} \geq L(p_\theta(x|z),q(z|x)) =\mathbf{E}_{p_d(x)}\left[\mathbf{E}_{q(z|x)}\left[\log p_\theta(x|z)\right]-\mathrm{KL}\left( q(z|x)||p(z)\right)\right].
  \end{eqnarray*}
  任意选择 $q(z|x)$ 都能给出一个有效的下界。变分自编码器把变分后验 $q(z|x)$ 替换成一个推理网络 $q_{\phi}(z|x)$，它与 $p_{\theta}(x|z)$ 一起训练，共同最大化 $L(p_\theta,q_\phi)$。

变分后验 $q_{\phi}(z|x)$ 也叫**编码器（encoder）**，生成模型 $p_{\theta}(x|z)$ 叫**解码器（decoder）**或生成器。

第一项 $\mathbf{E}_{q(z|x)}\left[\log p_\theta(x|z)\right]$ 是负的重构误差。确实，在高斯假设下，即 $p_{\theta}(x|z) = \mathcal{N}(\mu_{\theta}(z), I)$，$\log p_\theta(x|z)$ 这一项约化为 $\propto \|x-\mu_\theta(z)\|^2$，这也是实践中常用的形式。$\mathrm{KL}\left( q(z|x)||p(z)\right)$ 这一项可以看作正则化项，它要求变分后验 $q_\phi(z|x)$ 与先验 $p(z)= \mathcal{N}(0, I)$ 匹配。

变分自编码器由 [Kingma 和 Welling（2013）](https://arxiv.org/abs/1312.6114)提出，也可以看 [(Doersch, 2016)](https://arxiv.org/abs/1606.05908) 的教程。

PyTorch 里有各种 VAE 例子，见[这里](https://github.com/pytorch/examples/tree/master/vae)或[这里](https://github.com/yunjey/pytorch-tutorial/blob/master/tutorials/03-advanced/variational_autoencoder/main.py#L38-L65)。下面的代码取自最后一个来源。

![变分自编码器。](vae.png)


In [ ]:
# 超参数
image_size = 784
h_dim = 400
z_dim = 20
num_epochs = 15
learning_rate = 1e-3

# VAE 模型
class VAE(nn.Module):
    def __init__(self, image_size=784, h_dim=400, z_dim=20):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(image_size, h_dim)
        self.fc2 = nn.Linear(h_dim, z_dim)
        self.fc3 = nn.Linear(h_dim, z_dim)
        self.fc4 = nn.Linear(z_dim, h_dim)
        self.fc5 = nn.Linear(h_dim, image_size)
        
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h), self.fc3(h)
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(log_var/2)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc4(z))
        return torch.sigmoid(self.fc5(h))
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_reconst = self.decode(z)
        return x_reconst, mu, log_var

model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

这里损失的重构部分不用 MSE，而是用二元交叉熵。下面的代码仍然来自 PyTorch 教程（做了一点小修改以避免警告！）。


In [ ]:
# 开始训练
for epoch in range(num_epochs):
    for i, (x, _) in enumerate(data_loader):
        # 前向传播
        x = x.to(device).view(-1, image_size)
        x_reconst, mu, log_var = model(x)
        
        # 计算重构损失和 KL 散度
        # 高斯之间的 KL 散度，见 VAE 论文附录 B 或 (Doersch, 2016)：
        # https://arxiv.org/abs/1606.05908
        reconst_loss = F.binary_cross_entropy(x_reconst, x, reduction='sum')
        kl_div = - 0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        
        # 反向传播并优化
        loss = reconst_loss + kl_div
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 10 == 0:
            print ("Epoch[{}/{}], Step [{}/{}], Reconst Loss: {:.4f}, KL Div: {:.4f}" 
                   .format(epoch+1, num_epochs, i+1, len(data_loader), reconst_loss.item()/len(x), kl_div.item()/len(x)))

看看我们的网络是如何重构最后一个 batch 的。我们成对展示原始数字和重构版本。


In [ ]:
plot_reconstruction(model)

现在看看我们的网络如何生成新样本。


In [ ]:
plot_generation(model)

不太理想，但我们没有让网络训练太久……话说回来，我们对生成的数字没有控制力。在 notebook 的剩余部分，我们探索生成 0、1、2 等等数字的方法。


作为附带收获，我们展示如何借助下面描述的 Gumbel VAE 让我们的 VAE 做聚类。但在此之前，我们先稍微偷个懒……


# 用'条件'VAE 偷懒

我们先用标签（就像课程里用[条件 GAN](https://dataflowr.github.io/website/modules/10-generative-adversarial-networks/) 那样）。思路是稍微修改上面的架构：除了解码器计算出的编码之外，再把标签的 onehot 版本喂给解码器。

先写一个函数，把标签转成 onehot 编码。这个函数会在训练循环里用到（不是在神经网络架构里！）。


In [ ]:
n_classes = 10
def l_2_onehot(labels,nb_digits=n_classes):
    # 接收标签（来自 dataloader），返回 onehot 编码后的标签
    #
    # 你的代码
    #

你可以在一个 batch 上测试它。


In [ ]:
(x,labels) = next(iter(data_loader))

In [ ]:
labels

In [ ]:
l_2_onehot(labels)

现在修改 VAE 的架构：解码器的输入是随机编码与标签 onehot 编码的拼接。


In [ ]:
n_classes = 10

class VAE_Cond(nn.Module):
    def __init__(self, image_size=784, h_dim=400, z_dim=20, n_classes = 10):
        super(VAE_Cond, self).__init__()
        self.fc1 = nn.Linear(image_size, h_dim)
        self.fc2 = nn.Linear(h_dim, z_dim)
        self.fc3 = nn.Linear(h_dim, z_dim)
        #
        # 你的代码
        #
        
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h), self.fc3(h)
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(log_var/2)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, l_onehot):
        #
        # 你的代码 / 用 torch.cat
        #      
    
    def forward(self, x, l_onehot):
        #
        # 你的代码 / 用 F.gumbel_softmax
        #

在一个 batch 上测试你的新模型：


In [ ]:
model_C = VAE_Cond().to(device)
x = x.to(device).view(-1, image_size)
l_onehot = l_2_onehot(labels)
l_onehot = l_onehot.to(device)
model_C(x, l_onehot)

现在你可以修改网络的训练循环了。参数 $\beta$ 让你可以缩放损失中的 KL 项，正如 [$\beta$-VAE 论文](https://openreview.net/forum?id=Sy2fzU9gl) 中所解释的，见论文公式 (4)。


In [ ]:
def train_C(model, data_loader=data_loader,num_epochs=num_epochs, beta=10., verbose=True):
    nmi_scores = []
    model.train(True)
    for epoch in range(num_epochs):
        for i, (x, labels) in enumerate(data_loader):
            # 前向传播
            x = x.to(device).view(-1, image_size)
            #
            # 你的代码
            #
            
            
            reconst_loss = F.binary_cross_entropy(x_reconst, x, reduction='sum')
            kl_div =  - 0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
            
            # 反向传播并优化
            loss = reconst_loss + beta*kl_div
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if verbose:
                if (i+1) % 10 == 0:
                    print("Epoch[{}/{}], Step [{}/{}], Reconst Loss: {:.4f}, KL Div: {:.4f}"
                           .format(epoch+1, num_epochs, i+1, len(data_loader), reconst_loss.item()/len(x),
                                   kl_div.item()/len(x)))

In [ ]:
model_C = VAE_Cond().to(device)
optimizer = torch.optim.Adam(model_C.parameters(), lr=learning_rate)

In [ ]:
train_C(model_C,num_epochs=15,verbose=True)

In [ ]:
plot_conditional_generation(model_C, n=8)

这里你应该能得到不错的结果。现在我们不用标签了……


# 用 Gumbel VAE 不偷懒

实现一个 VAE，加入一个类别变量 $c\in \{0,\dots 9\}$，使潜变量模型变成 $p(c,z,x) = p(c)p(z)p_{\theta}(x|c,z)$，变分后验是 $q_{\phi}(c|x)q_{\phi}(z|x)$，正如这篇 NeurIPS 论文所述：[(Dupont, 2018)](https://arxiv.org/abs/1804.00104)。对之前的架构做最小的修改即可。

思路是在潜空间里加入一个类别变量。你希望这个类别变量能编码数字的类别，这样网络可以用它来做更好的重构。而且，如果一切按计划进行，你就能按类别条件生成数字：借助潜类别变量 $c$ 选择类别，然后从这个类别生成数字。

正如上面提到的，为了既能采样随机变量、又能继续使用反向传播，我们需要重参数化技巧，这对高斯随机变量很容易。对于类别随机变量，重参数化技巧在 [(Jang et al., 2016)](https://arxiv.org/abs/1611.01144) 中有解释。在 PyTorch 中，这由 [F.gumbel_softmax](https://pytorch.org/docs/stable/nn.html?highlight=gumbel_softmax#torch.nn.functional.gumbel_softmax) 实现。


In [ ]:
n_classes = 10

class VAE_Gumbel(nn.Module):
    def __init__(self, image_size=784, h_dim=400, z_dim=20, n_classes = 10):
        super(VAE_Gumbel, self).__init__()
        #
        # 你的代码
        #
        
    def encode(self, x):
        #
        # 你的代码 / 用 F.log_softmax
        #
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(log_var/2)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_onehot):
        #
        # 你的代码 / 用 torch.cat
        #
    
    def forward(self, x):
        #
        # 你的代码 / 用 F.gumbel_softmax
        #

In [ ]:
model_G = VAE_Gumbel().to(device)
optimizer = torch.optim.Adam(model_G.parameters(), lr=learning_rate)

你需要修改损失，把类别随机变量考虑进去，先验是在 $\{0,\dots 9\}$ 上的均匀分布，见 [(Dupont, 2018)](https://arxiv.org/abs/1804.00104) 附录 A.2


In [ ]:
def train_G(model, data_loader=data_loader,num_epochs=num_epochs, beta = 1., verbose=True):
    nmi_scores = []
    model.train(True)
    for epoch in range(num_epochs):
        all_labels = []
        all_labels_est = []
        for i, (x, labels) in enumerate(data_loader):
            # 前向传播
            x = x.to(device).view(-1, image_size)
            #
            # 你的代码
            #
            
            reconst_loss = F.binary_cross_entropy(x_reconst, x, reduction='sum')
            #
            # 你的代码
            #

            # 反向传播并优化
            loss = # loss = # 你的代码
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if verbose:
                if (i+1) % 10 == 0:
                    print ("Epoch[{}/{}], Step [{}/{}], Reconst Loss: {:.4f}, KL Div: {:.4f}, Entropy: {:.4f}" 
                           .format(epoch+1, num_epochs, i+1, len(data_loader), reconst_loss.item()/len(x),
                                   kl_div.item()/len(x), H_cat.item()/len(x)))

In [ ]:
train_G(model_G,num_epochs=10,verbose=True)

In [ ]:
plot_reconstruction(model_G)

这是重构的部分，但我们更关心生成。对每个类别，我们借助 `plot_conditional_generation()` 函数生成 8 个样本。


In [ ]:
plot_conditional_generation(model_G, n=8)

看起来我们最初的想法没有起作用……

发生的情况是，网络没有使用类别变量。我们可以追踪真实标签与网络预测标签之间的[归一化互信息（NMI）](https://en.wikipedia.org/wiki/Mutual_information#Normalized_variants)（见 [scikit-learn 里的这个方法](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.normalized_mutual_info_score.html)），网络预测标签就取概率最大的那个类别。

修改你的训练循环，让它返回每个 epoch 的归一化互信息（NMI）。画出曲线，确认 NMI 确实在下降。


这个问题在 [(Burgess et al., 2018)](https://arxiv.org/abs/1804.03599) 中有解释，第 5 节提出了一个解决方案。

为了强制网络使用类别变量，我们按照 [(Dupont, 2018)](https://arxiv.org/abs/1804.00104) 第 3 节公式 (7) 修改损失。

在训练循环里实现这个改动，并画出新的 NMI 曲线。取 $\beta = 20, C_z=100, C_c=100$ 时，你应该能看到 NMI 上升。


In [ ]:
model_G = VAE_Gumbel().to(device)
optimizer = torch.optim.Adam(model_G.parameters(), lr=learning_rate)

In [ ]:
def train_G_modified_loss(model, data_loader=data_loader,num_epochs=num_epochs, beta=1. , C_z_fin=0, C_c_fin=0, verbose=True):
    #
    # 你的代码
    #
    return nmi_scores

In [ ]:
# 超参数
num_epochs = 20
learning_rate = 1e-3
beta = 20
C_z_fin=100
C_c_fin=100

model_G = VAE_Gumbel(z_dim = z_dim).to(device)
optimizer = torch.optim.Adam(model_G.parameters(), lr=learning_rate)

nmi = train_G_modified_loss(model_G, data_loader, num_epochs=num_epochs, beta=beta, C_z_fin=C_z_fin, C_c_fin=C_c_fin)

In [ ]:
plt.plot(nmi)

In [ ]:
plot_reconstruction(model_G)

In [ ]:
plot_conditional_generation(model_G, fix_number=None)

In [ ]:
plot_conditional_generation(model_G, fix_number=2)